# 🔗 Store Intelligence — Person Re-Identification Model

Train a lightweight **OSNet-x0.25** model to generate 128-d person embeddings
for cross-camera Re-ID. Uses triplet loss with online hard mining.

**Prerequisites:** Run `build_reid_dataset.py` locally to create the identity dataset.

**Output:** `training/models/weights/reid_osnet.pth`

In [ ]:
!pip install -q torch torchvision tqdm matplotlib scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

PROJECT_ROOT = Path('/content/drive/MyDrive/purplle_hackathon')
REID_DIR = PROJECT_ROOT / 'training' / 'data' / 'reid_dataset'
WEIGHTS_DIR = PROJECT_ROOT / 'training' / 'models' / 'weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Scan dataset
identities = sorted([d for d in REID_DIR.iterdir() if d.is_dir()]) if REID_DIR.exists() else []
total_crops = sum(len(list(d.glob('*.jpg'))) for d in identities)
print(f'Identities: {len(identities)}, Total crops: {total_crops}')
for d in identities[:5]:
    print(f'  {d.name}: {len(list(d.glob("*.jpg")))} crops')

In [ ]:
# OSNet-x0.25 Architecture (lightweight Re-ID backbone)
class ConvBNReLU(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x): return self.relu(self.bn(self.conv(x)))

class ChannelGate(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, mid), nn.ReLU(inplace=True),
            nn.Linear(mid, channels), nn.Sigmoid()
        )
    def forward(self, x): return x * self.gate(x).unsqueeze(-1).unsqueeze(-1)

class LightBottleneck(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        mid = out_c // 4
        self.conv1 = ConvBNReLU(in_c, mid, 1, 1, 0)
        self.conv2 = ConvBNReLU(mid, mid, 3, 1, 1, groups=mid)  # depthwise
        self.conv3 = nn.Sequential(nn.Conv2d(mid, out_c, 1, bias=False), nn.BatchNorm2d(out_c))
        self.gate = ChannelGate(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity() if in_c == out_c else nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, bias=False), nn.BatchNorm2d(out_c))
    def forward(self, x):
        out = self.conv3(self.conv2(self.conv1(x)))
        out = self.gate(out)
        return self.relu(out + self.shortcut(x))

class OSNet025(nn.Module):
    """Simplified OSNet-x0.25 for 128-d Re-ID embeddings."""
    def __init__(self, embed_dim=128):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBNReLU(3, 16, 7, 2, 3),
            nn.MaxPool2d(3, 2, 1),
        )
        self.layer1 = nn.Sequential(LightBottleneck(16, 64), LightBottleneck(64, 64))
        self.pool1 = nn.Sequential(ConvBNReLU(64, 64, 1, 1, 0), nn.AvgPool2d(2, 2))
        self.layer2 = nn.Sequential(LightBottleneck(64, 96), LightBottleneck(96, 96))
        self.pool2 = nn.Sequential(ConvBNReLU(96, 96, 1, 1, 0), nn.AvgPool2d(2, 2))
        self.layer3 = nn.Sequential(LightBottleneck(96, 128), LightBottleneck(128, 128))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, embed_dim)
        self.bn = nn.BatchNorm1d(embed_dim)

    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(self.layer1(x))
        x = self.pool2(self.layer2(x))
        x = self.layer3(x)
        x = self.gap(x).flatten(1)
        x = self.bn(self.fc(x))
        return F.normalize(x, p=2, dim=1)

model = OSNet025(embed_dim=128).to(device)
params = sum(p.numel() for p in model.parameters())
print(f'OSNet-x0.25 created: {params:,} parameters')

In [ ]:
# Triplet Dataset with online hard mining
class TripletReIDDataset(Dataset):
    def __init__(self, root_dir, transform=None, samples_per_id=4):
        self.root = Path(root_dir)
        self.transform = transform
        self.samples_per_id = samples_per_id
        self.id_dirs = sorted([d for d in self.root.iterdir() if d.is_dir()])
        self.id_to_images = {}
        for d in self.id_dirs:
            imgs = sorted(d.glob('*.jpg'))
            if len(imgs) >= 2:
                self.id_to_images[d.name] = imgs
        self.id_list = list(self.id_to_images.keys())
        print(f'TripletDataset: {len(self.id_list)} identities')

    def __len__(self):
        return len(self.id_list) * self.samples_per_id

    def __getitem__(self, idx):
        # Sample anchor identity
        anchor_id = self.id_list[idx % len(self.id_list)]
        anchor_imgs = self.id_to_images[anchor_id]
        # Anchor + Positive (same identity)
        idxs = np.random.choice(len(anchor_imgs), 2, replace=len(anchor_imgs)<2)
        anchor = Image.open(anchor_imgs[idxs[0]]).convert('RGB')
        positive = Image.open(anchor_imgs[idxs[1]]).convert('RGB')
        # Negative (different identity)
        neg_id = anchor_id
        while neg_id == anchor_id:
            neg_id = np.random.choice(self.id_list)
        neg_imgs = self.id_to_images[neg_id]
        negative = Image.open(neg_imgs[np.random.randint(len(neg_imgs))]).convert('RGB')

        if self.transform:
            anchor = self.transform(anchor)
            positive = self.transform(positive)
            negative = self.transform(negative)
        return anchor, positive, negative

print('TripletReIDDataset defined')

In [ ]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomCrop((256, 128), padding=10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

dataset = TripletReIDDataset(str(REID_DIR), transform=train_transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2, drop_last=True)
print(f'DataLoader: {len(loader)} batches')

In [ ]:
# Training setup
EPOCHS = 30
criterion = nn.TripletMarginLoss(margin=0.3)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)

# Warmup + cosine scheduler
warmup_epochs = 5
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

losses = []
print(f'Training for {EPOCHS} epochs...')

In [ ]:
from tqdm import tqdm

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for anchor, positive, negative in tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False):
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        embed_a = model(anchor)
        embed_p = model(positive)
        embed_n = model(negative)

        loss = criterion(embed_a, embed_p, embed_n)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS} — loss={avg_loss:.4f} lr={optimizer.param_groups[0]["lr"]:.6f}')

print('Training complete!')

In [ ]:
# Plot loss curve
plt.figure(figsize=(10, 4))
plt.plot(losses, color='#2196F3', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Triplet Loss')
plt.title('Re-ID Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate: t-SNE visualization of embeddings
from sklearn.manifold import TSNE

model.eval()
eval_transform = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

all_embeds, all_ids = [], []
for i, d in enumerate(identities[:20]):  # First 20 identities
    imgs = sorted(d.glob('*.jpg'))[:10]
    for img_path in imgs:
        img = Image.open(img_path).convert('RGB')
        tensor = eval_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            embed = model(tensor).cpu().numpy().flatten()
        all_embeds.append(embed)
        all_ids.append(i)

if len(all_embeds) > 10:
    embeds = np.array(all_embeds)
    ids = np.array(all_ids)
    perp = min(30, len(embeds) - 1)
    tsne = TSNE(n_components=2, perplexity=max(5, perp), random_state=42)
    coords = tsne.fit_transform(embeds)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(coords[:,0], coords[:,1], c=ids, cmap='tab20', s=30, alpha=0.7)
    plt.colorbar(scatter, label='Identity ID')
    plt.title('t-SNE of Re-ID Embeddings (color = identity)')
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough embeddings for t-SNE visualization')

In [ ]:
# CMC Rank-1 and Rank-5 accuracy
if len(all_embeds) > 5:
    from sklearn.metrics import pairwise_distances
    dists = pairwise_distances(embeds, metric='euclidean')
    np.fill_diagonal(dists, np.inf)

    rank1, rank5, total = 0, 0, 0
    for i in range(len(embeds)):
        sorted_idx = np.argsort(dists[i])
        query_id = ids[i]
        top1_id = ids[sorted_idx[0]]
        top5_ids = ids[sorted_idx[:5]]
        if top1_id == query_id: rank1 += 1
        if query_id in top5_ids: rank5 += 1
        total += 1

    print(f'CMC Rank-1: {rank1/total:.3f}')
    print(f'CMC Rank-5: {rank5/total:.3f}')
else:
    print('Not enough data for CMC evaluation')

In [ ]:
# Save model
save_path = WEIGHTS_DIR / 'reid_osnet.pth'
torch.save(model.state_dict(), str(save_path))
print(f'Re-ID model saved to: {save_path}')

## ✅ Summary

- OSNet-x0.25 trained with triplet loss for 128-d person embeddings
- Model saved to `training/models/weights/reid_osnet.pth`
- Embeddings enable cross-camera person re-identification

**Next:** Run `04_full_pipeline.ipynb` for end-to-end integration.